# Mistral

In [1]:
import os
import torch
import gc
import warnings

from datasets import load_dataset
from lightning import Trainer
from lightning.pytorch import LightningDataModule, LightningModule
from lightning.pytorch.callbacks import TQDMProgressBar
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, PeftModel

warnings.filterwarnings("ignore")
os.environ["TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL"] = "0"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

/home/megad/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:814: UserWarning: Can't initialize amdsmi - Error code: 34
  warnings.warn(f"Can't initialize amdsmi - Error code: {e.err_code}")


In [2]:
model_name = "mistralai/Mistral-7B-Instruct-v0.3"

LORA_DIR = ".ipynb_checkpoints/mistral7b_lora"

os.makedirs(".ipynb_checkpoints", exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


# Data

In [3]:
def load_base_model():
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_double_quant=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=quant_config,
        device_map="auto",
        trust_remote_code=True,
    )

    return model

def attach_lora(model):
    lora_config = LoraConfig(
        r=8,
        lora_alpha=32,
        lora_dropout=0.1,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj"
        ],
        task_type="CAUSAL_LM"
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    return model

class DataModule(LightningDataModule):
    def __init__(self, batch_size=2, max_length=128):
        super().__init__()
        self.batch_size = batch_size
        self.max_length = max_length
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

    def setup(self, stage=None):
        dataset = load_dataset(
            "Despina/project_gutenberg",
            "fiction_books",
            split="train",
            streaming=True
        ).shuffle(seed=42, buffer_size=1000)

        self.train_dataset = dataset.take(500)

    def collate_fn(self, batch):
        texts = [x["text"] for x in batch]

        enc = self.tokenizer(
            texts,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        return enc["input_ids"], enc["input_ids"], enc["attention_mask"]

    def train_dataloader(self):
        return DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            collate_fn=self.collate_fn
        )
    
class QLoRAModule(LightningModule):
    def __init__(self):
        super().__init__()
        base = load_base_model()

        for param in base.parameters():
            param.requires_grad = False

        self.model = attach_lora(base)

    def forward(self, input_ids, labels=None, attention_mask=None):
        return self.model(
            input_ids=input_ids,
            labels=labels,
            attention_mask=attention_mask
        )

    def training_step(self, batch, batch_idx):
        input_ids, labels, attention_mask = batch
        outputs = self(input_ids, labels, attention_mask)
        loss = outputs.loss
        self.log("train_loss", loss)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=2e-4)

# Finetuner

In [4]:
def train_model(epochs=1, batch_size=2):

    data = DataModule(batch_size=batch_size)
    model = QLoRAModule()

    progress_bar = TQDMProgressBar(refresh_rate=20)

    trainer = Trainer(
        max_epochs=epochs,
        accelerator="auto",
        logger=False,
        callbacks=[progress_bar]
    )

    trainer.fit(model, datamodule=data)

    #SAVE ONLY LORA ADAPTER
    model.model.save_pretrained(LORA_DIR)

    print("LoRA adapter saved to:", LORA_DIR)

    del model
    gc.collect()
    torch.cuda.empty_cache()

In [5]:
#train_model(epochs=1, batch_size=2)

# Generation

In [6]:
#Load inference and check if model is already in memory
#This way we don't have to realod the weights 15 million times
def setup_inference():
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    base_model = load_base_model()

    if os.path.exists(LORA_DIR):
        print("Loading LoRA adapter")
        model = PeftModel.from_pretrained(base_model, LORA_DIR)
    else:
        print("No LoRA adapter found. Using base model")
        model = base_model

    model.eval()
    return model, tokenizer

#Preload once
print("Loading model")
inference_model, inference_tokenizer = setup_inference()


def generate(prompt, max_length=150):
    formatted = f"<s>[INST] {prompt} [/INST]"

    inputs = inference_tokenizer(formatted, return_tensors="pt").to(inference_model.device)

    output = inference_model.generate(
        **inputs,
        max_length=max_length,
        do_sample=True,
        temperature=0.7,
        top_k=50,
        repetition_penalty=1.2,
        pad_token_id=inference_tokenizer.eos_token_id,
    )

    return inference_tokenizer.decode(output[0], skip_special_tokens=True)


Loading model


Loading weights: 100%|██████████| 291/291 [00:25<00:00, 11.43it/s, Materializing param=model.norm.weight]                               


Loading LoRA adapter


### Generate prompts

In [ ]:
def show_generation(prompt):

    #Force model to load BEFORE printing separators
    print("---------------------------")
    print("PROMPT")
    result = generate(prompt)
    print(result)

In [8]:
#Generate multiple prompts
show_generation("Tell me about a brave warrior.")
show_generation("Describe a brave knight in medieval times.")

---------------------------
Tell me about a brave warrior.  In the annals of history, there are numerous tales of brave warriors who have made indelible marks on their respective epochs. I'd like to share with you one such story: Richard Lionheart, also known as King Richard I of England and Richard Coeur de Lion (Richard Heart of Lions).

Born in Oxfordshire, England, on September 8th, 1157, he was not only known for his bravery but also for his leadership qualities that inspired allegiance from many. He ascended to the throne after the death of his father Henry II during the Crusades. Despite spending very little time at home due to
---------------------------
Describe a brave knight in medieval times.  In the heart of medieval Europe, during a time when castles and chivalry were a way of life, there stood a figure who embodied both valor and nobility - Sir Lancelot du Lac, one of the most celebrated knights of all time.

Born to King Bors de Talis at Camelot, he was destined for gre